In [ ]:
import math
import re
from collections import Counter
from pathlib import Path
from urllib.parse import urlsplit

import pandas as pd
import tldextract

In [51]:
PROCESSED_DATA_PATH = Path("../data/train_preprocessed.csv")

In [52]:
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", None)

In [53]:
df = pd.read_csv("../data/train.csv")

# Normalizing urls

In [54]:
SCHEME_PATTERN = re.compile(r"^[a-z][a-z0-9+.-]*://", re.IGNORECASE)

In [55]:
def normalize_url(url: str) -> str:
    """Normalize url: case, scheme, www., trailing slash."""
    url = url.strip().lower()
    url = SCHEME_PATTERN.sub("", url, count=1)

    if url.startswith("www."):
        url = url[4:]

    return url.rstrip("/")


def parse_url(url: str):
    """Parse scheme-less URLs as URLs rather than paths."""
    url = url.strip()
    if not SCHEME_PATTERN.match(url):
        url = f"//{url}"

    try:
        return urlsplit(url)
    except ValueError:
        return urlsplit("")

In [56]:
parsed = df["url"].map(parse_url)

df = df.assign(
    normalized_url=df["url"].map(normalize_url),
)

In [57]:
audit_df = df.copy()

duplicate_summary = pd.DataFrame(
    [
        {
            "definition": "Exact URL",
            "duplicate_rows": audit_df["url"].duplicated().sum(),
            "rows_in_duplicate_groups": audit_df["url"].duplicated(keep=False).sum(),
            "duplicate_groups": (audit_df.groupby("url").size() > 1).sum(),
            "groups_with_conflicting_labels": (audit_df.groupby("url")["label"].nunique() > 1).sum(),
        },
        {
            "definition": "Normalized URL",
            "duplicate_rows": audit_df["normalized_url"].duplicated().sum(),
            "rows_in_duplicate_groups": audit_df["normalized_url"].duplicated(keep=False).sum(),
            "duplicate_groups": (audit_df.groupby("normalized_url").size() > 1).sum(),
            "groups_with_conflicting_labels": (audit_df.groupby("normalized_url")["label"].nunique() > 1).sum(),
        },
    ]
)

duplicate_summary

,definition,duplicate_rows,rows_in_duplicate_groups,duplicate_groups,groups_with_conflicting_labels
0,Exact URL,80,160,80,50
1,Normalized URL,276,521,245,50


There are more duplications based on the normalized URLs.

If labels are in conflict drop all occurances of the URL, otherwise keep first.

In [58]:
def deduplicate_urls(df: pd.DataFrame) -> pd.DataFrame:
    """Remove duplicate URLS.

    Remove duplicate URLs with conflicting labels completely, keeping the first occurrence of other URLs.
    """
    duplicated_urls = df[df["normalized_url"].duplicated(keep=False)]
    grouped = duplicated_urls.groupby(["normalized_url", "label"]).size().reset_index(name="count")
    different_labels = grouped[grouped.duplicated(subset=["normalized_url"], keep=False)]
    df = df[~df["normalized_url"].isin(different_labels["normalized_url"])]
    df = df.drop_duplicates(subset=["normalized_url"], keep="first")

    return df

In [59]:
df = deduplicate_urls(df)

In [60]:
df["label"].value_counts()

label
0    4257
1    4068
Name: count, dtype: int64

# Feature engineering

In [61]:
def entropy(s):
    """Calculate entropy of a string.

    Return zero for empty strings or strings with only one unique character.
    """
    if not s:
        return 0

    counts = Counter(s)
    n = len(s)

    if len(counts) <= 1:
        return 0

    return -sum((count / n) * math.log2(count / n) for count in counts.values())

In [62]:
def normalized_entropy(s):
    """Calculate the normalized entropy of a string.

    Return zero for empty strings or strings with only one unique character.
    """
    if not s:
        return 0

    counts = Counter(s)
    if len(counts) <= 1:
        return 0

    n = len(s)
    h = -sum((count / n) * math.log2(count / n) for count in counts.values())

    return h / math.log2(len(counts))

In [66]:
# basic feat eng redone
import ipaddress

TLD_EXTRACT = tldextract.TLDExtract(suffix_list_urls=None)

parsed = df["url"].map(parse_url)
hostname = parsed.map(lambda value: (value.hostname or "").lower())
path = parsed.map(lambda value: value.path)
query = parsed.map(lambda value: value.query)
fragment = parsed.map(lambda value: value.fragment)
extracted = hostname.map(TLD_EXTRACT)


def is_ip_literal(host: str) -> bool:
    """Check if the host is an IP address literal."""
    try:
        ipaddress.ip_address(host.strip("[]"))
        return True
    except ValueError:
        return False


def get_port(parsed_url) -> int:
    """Get the port from a parsed URL, returning 0 if not specified or invalid."""
    try:
        return parsed_url.port or 0
    except ValueError:
        return 0


ports = parsed.map(get_port)


df = df.assign(
    # Parsed URL components
    domain=extracted.map(lambda value: value.domain),
    subdomain=extracted.map(lambda value: value.subdomain),
    suffix=extracted.map(lambda value: value.suffix),
    registered_domain=extracted.map(lambda value: value.top_domain_under_public_suffix),
    scheme=parsed.map(lambda value: value.scheme.lower()),
    hostname=hostname,
    path=path,
    query=query,
    fragment=fragment,
    # Existing / corrected structural features
    query_parameters=query.map(lambda value: value.count("&") + 1 if value else 0),
    subdomain_labels=extracted.map(lambda value: len(value.subdomain.split(".")) if value.subdomain else 0),
    # Host and parsing features
    no_host=hostname.eq(""),
    is_ip_literal_host=hostname.map(is_ip_literal),
    host_label_count=hostname.map(lambda value: len([part for part in value.split(".") if part])),
    has_port=ports.gt(0),
    port=ports,
    unusual_port=ports.ne(0) & ~ports.isin([80, 443]),
    # Path / query structure
    path_segment_count=path.map(lambda value: len([part for part in value.split("/") if part])),
    slash_count=df["url"].str.count("/", flags=0),
    underscore_count=df["url"].str.count("_", flags=0),
    equals_count=df["url"].str.count("=", flags=0),
    ampersand_count=df["url"].str.count("&", flags=0),
    non_alphanumeric_count=df["url"].str.count(r"[^A-Za-z0-9]"),
    # URL-wide signals you already partly had
    digit_ratio=df["url"].map(lambda value: sum(char.isdigit() for char in value) / max(len(value), 1)),
    has_at=df["url"].str.contains("@", regex=False),
    has_percent_encoding=df["url"].str.contains(r"%[0-9A-Fa-f]{2}", regex=True),
    has_punycode=hostname.str.contains("xn--", regex=False),
    has_unicode=df["url"].map(lambda value: any(ord(char) > 127 for char in value)),
    has_repeated_separator=df["url"].str.contains(r"//|__|==|&&|\.\.|--", regex=True),
    double_slash_in_path=path.str.contains("//", regex=False),
    # Component-level ratios
    host_digit_ratio=hostname.str.count(r"\d") / hostname.str.len().clip(lower=1),
    host_letter_ratio=hostname.str.count(r"[A-Za-z]") / hostname.str.len().clip(lower=1),
    path_digit_ratio=path.str.count(r"\d") / path.str.len().clip(lower=1),
    path_letter_ratio=path.str.count(r"[A-Za-z]") / path.str.len().clip(lower=1),
    query_digit_ratio=query.str.count(r"\d") / query.str.len().clip(lower=1),
    query_letter_ratio=query.str.count(r"[A-Za-z]") / query.str.len().clip(lower=1),
)

In [67]:
# entropy
entropy_features = ["url", "hostname", "domain", "subdomain", "path", "query", "fragment"]

for feature in entropy_features:
    values = df[feature].fillna("")
    norm_entropy_lookup = {value: normalized_entropy(value) for value in values.unique()}
    entropy_lookup = {value: entropy(value) for value in values.unique()}
    df[f"{feature}_norm_entropy"] = values.map(norm_entropy_lookup)
    df[f"{feature}_entropy"] = values.map(entropy_lookup)

In [68]:
# length
length_features = ["url", "hostname", "domain", "subdomain", "path", "query", "fragment"]

for feature in length_features:
    df[f"{feature}_length"] = df[feature].str.len()

In [69]:
df[["subdomain_entropy", "subdomain_norm_entropy"]].corr()

,subdomain_entropy,subdomain_norm_entropy
subdomain_entropy,1.000000,0.930497
subdomain_norm_entropy,0.930497,1.000000


In [70]:
# save preprocessed training data
df.to_csv(PROCESSED_DATA_PATH, index=False)